# Análise de negócio com SQL

In [0]:

%sql

SELECT
    COUNT(*) AS linhas_venda,
    COUNT(DISTINCT id_pedido) AS pedidos_distintos,
    COUNT(DISTINCT id_cliente) AS clientes_distintos,
    COUNT(DISTINCT id_produto) AS produtos_distintos,
    ROUND(SUM(valor_venda), 2) AS valor_total_vendas,
    SUM(quantidade_vendida) AS unidades_vendidas,
    ROUND(AVG(valor_venda), 2) AS valor_medio_por_linha
FROM first_catalog.default.fact_vendas;

linhas_venda,pedidos_distintos,clientes_distintos,produtos_distintos,valor_total_vendas,unidades_vendidas,valor_medio_por_linha
51290,25035,1590,10292,12642502.22,178312,246.49


### Analisar as vendas por país
A consulta abaixo ajuda a identificar os países com maior valor de vendas. O GROUP BY agrupa as linhas por país e o ORDER BY ordena o resultado pelo total calculado.

In [0]:

%sql

SELECT
    pais,
    COUNT(*) AS linhas_venda,
    COUNT(DISTINCT id_pedido) AS pedidos,
    ROUND(SUM(valor_venda), 2) AS valor_total_vendas,
    SUM(quantidade_vendida) AS unidades_vendidas
FROM first_catalog.default.fact_vendas
GROUP BY pais
ORDER BY valor_total_vendas DESC
LIMIT 15;

pais,linhas_venda,pedidos,valor_total_vendas,unidades_vendidas
United States,9251,4588,2545193.00,34688
Australia,3948,1982,1049659.45,14810
France,3921,1925,999163.92,14660
China,2592,1264,696502.21,9770
India,2350,1209,690835.72,8962
Germany,2108,1051,560179.74,7971
Mexico,2281,1145,551354.61,8406
United Kingdom,1920,967,527489.67,7234
Indonesia,1784,895,487191.43,6870
Spain,1061,557,278308.02,4021


### Analisar as categorias de produtos
Com a consulta abaixo podemos descobrir que categorias e subcategorias contribuem mais para o valor total das vendas.

In [0]:

%sql

SELECT
    categoria,
    subcategoria,
    COUNT(*) AS linhas_venda,
    ROUND(SUM(valor_venda), 2) AS valor_total_vendas,
    SUM(quantidade_vendida) AS unidades_vendidas,
    ROUND(AVG(valor_venda), 2) AS valor_medio_por_linha
FROM first_catalog.default.fact_vendas
GROUP BY categoria, subcategoria
ORDER BY valor_total_vendas DESC;

categoria,subcategoria,linhas_venda,valor_total_vendas,unidades_vendidas,valor_medio_por_linha
Tecnologia,Phones,3357,1706824.18,11870,508.44
Tecnologia,Copiers,2223,1509436.31,7454,679.01
Moveis,Chairs,3434,1501681.78,12336,437.30
Moveis,Bookcases,2411,1466572.25,8310,608.28
Material de Escritorio,Storage,5059,1127085.88,16917,222.79
Material de Escritorio,Appliances,1755,1011064.31,6078,576.11
Tecnologia,Machines,1486,779060.10,4906,524.27
Moveis,Tables,861,757041.94,3083,879.26
Tecnologia,Accessories,3075,749237.04,10946,243.65
Material de Escritorio,Binders,6155,461931.19,21433,75.05


### Analisar a evolução mensal
Consulta como a base para um gráfico de evolução mensal. Ordenamos por ano e mês, em vez de ordenar pelo nome do mês, para manter a sequência cronológica correta

In [0]:

%sql

SELECT
    YEAR(data_pedido) AS ano,
    MONTH(data_pedido) AS mes,
    COUNT(*) AS linhas_venda,
    ROUND(SUM(valor_venda), 2) AS valor_total_vendas,
    SUM(quantidade_vendida) AS unidades_vendidas
FROM first_catalog.default.fact_vendas
GROUP BY
    YEAR(data_pedido),
    MONTH(data_pedido)
ORDER BY
    ano,
    mes;

ano,mes,linhas_venda,valor_total_vendas,unidades_vendidas
2011,1,439,99905.62,1466
2011,2,362,90353.89,1188
2011,3,533,145115.70,1821
2011,4,572,118317.65,2041
2011,5,560,144176.13,1981
2011,6,917,215879.58,3109
2011,7,489,113706.73,1754
2011,8,873,207013.40,3011
2011,9,1057,290324.43,3718
2011,10,775,200075.32,2736


### crescimento anual em SQL
Calcular o total de vendas por ano e a variação percentual relativamente ao ano anterior. Uma análise é muito útil em entrevistas e em dashboards executivos

In [0]:

%sql

WITH vendas_anuais AS (
    SELECT
        YEAR(data_pedido) AS ano,
        SUM(valor_venda) AS valor_total_vendas
    FROM first_catalog.default.fact_vendas
    GROUP BY YEAR(data_pedido)
),
comparacao AS (
    SELECT
        ano,
        valor_total_vendas,
        LAG(valor_total_vendas) OVER (
            ORDER BY ano
        ) AS vendas_ano_anterior
    FROM vendas_anuais
)
SELECT
    ano,
    ROUND(valor_total_vendas, 2) AS valor_total_vendas,
    ROUND(vendas_ano_anterior, 2) AS vendas_ano_anterior,
    ROUND(
        100.0 * (valor_total_vendas - vendas_ano_anterior)
        / NULLIF(vendas_ano_anterior, 0),
        2
    ) AS crescimento_percentual
FROM comparacao
ORDER BY ano;

ano,valor_total_vendas,vendas_ano_anterior,crescimento_percentual
2011,2259450.95,null,null
2012,2677438.75,2259450.95,18.50
2013,3405746.55,2677438.75,27.20
2014,4299865.98,3405746.55,26.25


### analisar a eficiência logística
Analisar quantos dias os pedidos demoram a ser enviados e comparar os diferentes modos de envio.

In [0]:

%sql

SELECT
    modo_envio,
    COUNT(*) AS total_pedidos,
    ROUND(AVG(dias_para_envio), 2) AS media_dias_para_envio,
    MIN(dias_para_envio) AS minimo_dias,
    MAX(dias_para_envio) AS maximo_dias
FROM first_catalog.default.dim_pedido
GROUP BY modo_envio
ORDER BY media_dias_para_envio;

modo_envio,total_pedidos,media_dias_para_envio,minimo_dias,maximo_dias
Mesmo Dia,1319,0.04,0,1
Primeira Classe,3748,2.19,1,3
Segunda Classe,5015,3.21,2,5
Classe Padrao,14953,5.0,4,7


### a prioridade influencia o tempo de envio?
Os pedidos de prioridade alta são enviados mais rapidamente do que os de prioridade baixa?

In [0]:

%sql

SELECT
    prioridade_pedido,
    modo_envio,
    COUNT(*) AS total_pedidos,
    ROUND(AVG(dias_para_envio), 2) AS media_dias_para_envio
FROM first_catalog.default.dim_pedido
GROUP BY
    prioridade_pedido,
    modo_envio
ORDER BY
    prioridade_pedido,
    media_dias_para_envio;

prioridade_pedido,modo_envio,total_pedidos,media_dias_para_envio
Alto,Mesmo Dia,627,0.04
Alto,Primeira Classe,1716,2.11
Alto,Segunda Classe,1954,2.9
Alto,Classe Padrao,3341,4.26
Baixo,Classe Padrao,1175,6.49
Critico,Mesmo Dia,349,0.03
Critico,Primeira Classe,881,2.14
Critico,Segunda Classe,711,2.31
Medio,Mesmo Dia,343,0.05
Medio,Primeira Classe,1151,2.34


### preparar os KPIs do dashboard

In [0]:

%sql

WITH vendas AS (
    SELECT
        COUNT(*) AS linhas_venda,
        ROUND(SUM(valor_venda), 2) AS valor_total_vendas,
        SUM(quantidade_vendida) AS unidades_vendidas,
        COUNT(DISTINCT id_pedido) AS pedidos_com_vendas,
        COUNT(DISTINCT id_cliente) AS clientes_com_vendas,
        COUNT(DISTINCT id_produto) AS produtos_vendidos
    FROM first_catalog.default.fact_vendas
),
logistica AS (
    SELECT
        COUNT(*) AS total_pedidos,
        ROUND(AVG(dias_para_envio), 2) AS media_dias_para_envio
    FROM first_catalog.default.dim_pedido
)
SELECT
    v.valor_total_vendas,
    v.linhas_venda,
    v.pedidos_com_vendas,
    l.total_pedidos,
    v.clientes_com_vendas,
    v.produtos_vendidos,
    v.unidades_vendidas,
    l.media_dias_para_envio
FROM vendas v
CROSS JOIN logistica l;

valor_total_vendas,linhas_venda,pedidos_com_vendas,total_pedidos,clientes_com_vendas,produtos_vendidos,unidades_vendidas,media_dias_para_envio
12642502.22,51290,25035,25035,1590,10292,178312,3.96


### Criar a primeira visualização: vendas por país

In [0]:

%sql

SELECT
    pais,
    ROUND(SUM(valor_venda), 2) AS valor_total_vendas,
    COUNT(DISTINCT id_pedido) AS total_pedidos
FROM first_catalog.default.fact_vendas
GROUP BY pais
ORDER BY valor_total_vendas DESC
LIMIT 10;

pais,valor_total_vendas,total_pedidos
United States,2545193.00,4588
Australia,1049659.45,1982
France,999163.92,1925
China,696502.21,1264
India,690835.72,1209
Germany,560179.74,1051
Mexico,551354.61,1145
United Kingdom,527489.67,967
Indonesia,487191.43,895
Spain,278308.02,557


Databricks visualization. Run in Databricks to view.

### Criar a evolução mensal das vendas
O segundo gráfico vai mostrar como as vendas evoluíram ao longo do tempo, de 2011 a 2014.

In [0]:

%sql

SELECT
    YEAR(data_pedido) AS ano,
    MONTH(data_pedido) AS mes,
    DATE_FORMAT(data_pedido, 'yyyy-MM') AS ano_mes,
    ROUND(SUM(valor_venda), 2) AS valor_total_vendas,
    COUNT(DISTINCT id_pedido) AS total_pedidos
FROM first_catalog.default.fact_vendas
GROUP BY
    YEAR(data_pedido),
    MONTH(data_pedido),
    DATE_FORMAT(data_pedido, 'yyyy-MM')
ORDER BY
    ano,
    mes;

ano,mes,ano_mes,valor_total_vendas,total_pedidos
2011,1,2011-01,99905.62,213
2011,2,2011-02,90353.89,173
2011,3,2011-03,145115.70,273
2011,4,2011-04,118317.65,262
2011,5,2011-05,144176.13,291
2011,6,2011-06,215879.58,461
2011,7,2011-07,113706.73,245
2011,8,2011-08,207013.40,433
2011,9,2011-09,290324.43,522
2011,10,2011-10,200075.32,396


Databricks visualization. Run in Databricks to view.

### Vendas por Subcategoria
Identificar quais as subcategorias que mais contribuem para o valor total das vendas.

In [0]:

%sql

SELECT
    categoria,
    subcategoria,
    ROUND(SUM(valor_venda), 2) AS valor_total_vendas,
    SUM(quantidade_vendida) AS unidades_vendidas
FROM first_catalog.default.fact_vendas
GROUP BY
    categoria,
    subcategoria
ORDER BY
    valor_total_vendas DESC;

categoria,subcategoria,valor_total_vendas,unidades_vendidas
Tecnologia,Phones,1706824.18,11870
Tecnologia,Copiers,1509436.31,7454
Moveis,Chairs,1501681.78,12336
Moveis,Bookcases,1466572.25,8310
Material de Escritorio,Storage,1127085.88,16917
Material de Escritorio,Appliances,1011064.31,6078
Tecnologia,Machines,779060.10,4906
Moveis,Tables,757041.94,3083
Tecnologia,Accessories,749237.04,10946
Material de Escritorio,Binders,461931.19,21433


Databricks visualization. Run in Databricks to view.

### KPI — Valor total das vendas

In [0]:
%sql

SELECT
    ROUND(SUM(valor_venda), 2) AS valor_total_vendas
FROM first_catalog.default.fact_vendas;

valor_total_vendas
12642502.22


Databricks visualization. Run in Databricks to view.

### KPI - Total de pedidos

In [0]:
%sql

SELECT
    COUNT(DISTINCT id_pedido) AS total_pedidos
FROM first_catalog.default.fact_vendas;

total_pedidos
25035


Databricks visualization. Run in Databricks to view.

### KPI - Unidades vendidas

In [0]:
%sql

SELECT
    SUM(quantidade_vendida) AS unidades_vendidas
FROM first_catalog.default.fact_vendas;

unidades_vendidas
178312


Databricks visualization. Run in Databricks to view.

### KPI - Tempo médio até ao envio
Para este indicador, vamos usar a dimensão de pedidos, porque cada linha representa um pedido

In [0]:
%sql

SELECT
    ROUND(AVG(dias_para_envio), 2) AS media_dias_para_envio
FROM first_catalog.default.dim_pedido;

media_dias_para_envio
3.96


Databricks visualization. Run in Databricks to view.